# Manual Factuality Validation

Sanity-checks the automatic factuality decisions (`author_status`, `affiliation_status`) against a human reviewer who looks up each recommended persona directly on Semantic Scholar and OpenAlex.

In [ ]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote

import pandas as pd

RESULTS = Path('/data/datasets/LLMScholar-Personas/results')
SUMMARY_CSV   = RESULTS / 'summary_v2' / 'summary.csv'
FACT_FULL_CSV = RESULTS / 'summary_v2' / 'factuality_full.csv'
FACT_AFF_CSV  = RESULTS / 'summary_v2' / 'factuality_affiliation.csv'  # if Task 2 done
RESPONSES_DIR = RESULTS / 'responses'
OUT_DIR       = RESULTS / 'factualities_v2/manual'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV       = OUT_DIR / 'manual_validation_20_v2.csv'

RANDOM_STATE  = 42
N_REQUESTS    = 20
VALID_FLAGS   = ['cleaned', 'unchanged']

print(f'OUT_CSV = {OUT_CSV}')

In [ ]:
# 1. Sample 5 requests from summary.csv
df_summary = pd.read_csv(SUMMARY_CSV, low_memory=False)
print(f'summary.csv rows: {len(df_summary):,}')
df_valid = df_summary.query('valid_flag in @VALID_FLAGS').copy()
print(f'valid rows:        {len(df_valid):,}')
df_sample = df_valid.sample(N_REQUESTS, random_state=RANDOM_STATE).reset_index(drop=True)
df_sample.index.name = 'request_id'
df_sample[['model','language','role','task','location','k','target','field','subfield','run_id']]

In [ ]:
# 2. Locate each sampled request's JSON and extract its k recommendations.
def _candidate_dirs(language: str) -> list[Path]:
    return [d for d in RESPONSES_DIR.iterdir() if d.is_dir() and d.name.endswith(f'_{language}')]

def find_request(summary_row: pd.Series) -> tuple[Path | None, str | None, dict | None]:
    """Return (json_path, json_key, request_obj) matching the summary row.
    Walks all results_*_{language}/ directories and inspects each JSON to find
    the entry whose model + persona_context + user_request match exactly.
    """
    target = dict(
        model    = str(summary_row['model']),
        role     = str(summary_row['role']),
        task     = str(summary_row['task']),
        location = str(summary_row['location']),
        k        = int(summary_row['k']),
        f_target = str(summary_row['target']),
        field    = str(summary_row['field']),
        subfield = str(summary_row['subfield']),
    )
    for d in _candidate_dirs(summary_row['language']):
        for jpath in sorted(d.glob('*.json')):
            with open(jpath) as f:
                data = json.load(f)
            # Pre-filter: only inspect files whose first entry matches the model.
            sample_obj = next(iter(data.values()), {})
            if str(sample_obj.get('model','')) != target['model']:
                continue
            for key, obj in data.items():
                pc = obj.get('parameters', {}).get('persona_context', {})
                ur = obj.get('parameters', {}).get('user_request', {})
                if (str(pc.get('role','')) == target['role']
                    and str(pc.get('task','')) == target['task']
                    and str(pc.get('location','')) == target['location']
                    and int(ur.get('k', -1)) == target['k']
                    and str(ur.get('target','')) == target['f_target']
                    and str(ur.get('field','')) == target['field']
                    and str(ur.get('subfield','')) == target['subfield']):
                    return jpath, key, obj
    return None, None, None


def _extract_text(r: dict) -> str | None:
    """Pull the LLM textual content out of a response entry. Handles Gemini
    (response.candidates[].content.parts[].text), OpenAI batch
    (response.body.choices[].message.content), and Ollama (top-level
    message.content) shapes."""
    resp = r.get('response') if isinstance(r.get('response'), dict) else None
    if resp:
        cands = resp.get('candidates')
        if cands:
            parts = cands[0].get('content', {}).get('parts', [])
            if parts and parts[0].get('text'):
                return parts[0]['text']
        body = resp.get('body') if isinstance(resp.get('body'), dict) else None
        choices = (body or resp).get('choices') if isinstance(body or resp, dict) else None
        if choices:
            msg = choices[0].get('message', {})
            if msg.get('content'):
                return msg['content']
    msg = r.get('message') if isinstance(r.get('message'), dict) else None
    if msg and msg.get('content'):
        return msg['content']
    return None


def _strip_code_fence(s: str) -> str:
    s = s.strip()
    if s.startswith('```'):
        s = s.strip('`').strip()
        if s.lower().startswith('json'):
            s = s[4:].strip()
    return s


def _coerce_list(obj):
    """Accept [persona, …], {candidates: [...]}, {recommendations: [...]}, or a single persona dict."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ('candidates', 'recommendations', 'persons', 'people', 'results', 'items'):
            if isinstance(obj.get(key), list):
                return obj[key]
        if 'name' in obj or 'lastname' in obj:
            return [obj]
    return []


def extract_recommendations(req_obj: dict, run_id: int) -> list[dict]:
    responses = req_obj.get('responses', [])
    if not responses:
        return []
    idx = max(0, min(int(run_id) - 1, len(responses) - 1))
    txt = _extract_text(responses[idx])
    if not txt:
        return []
    try:
        return _coerce_list(json.loads(_strip_code_fence(txt)))
    except json.JSONDecodeError:
        return []


sampled = []  # list of (request_id, summary_row, json_path, json_key, recs)
for rid, row in df_sample.iterrows():
    jpath, jkey, req_obj = find_request(row)
    if req_obj is None:
        print(f'[request_id={rid}] WARNING: JSON not found for {row["model"]}/{row["language"]}')
        sampled.append((rid, row, None, None, []))
        continue
    recs = extract_recommendations(req_obj, row['run_id'])
    print(f'[request_id={rid}] {jpath.name} key={jkey} run_id={row["run_id"]} → {len(recs)} recommendations')
    sampled.append((rid, row, jpath, jkey, recs))

In [ ]:
# 3. Join with factuality_full.csv (and factuality_affiliation.csv if present) to get auto decisions.
JOIN_KEYS = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
USECOLS_FULL = JOIN_KEYS + ['author_status','oa_status','field_status','seniority_status','location_status',
                            'matched_name','researcher_id','match_score','gt_field',
                            'oa_id','oa_display_name','oa_match_score']
df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False, usecols=lambda c: c in USECOLS_FULL or c in JOIN_KEYS)
print(f'factuality_full rows: {len(df_full):,}')

aff_lookup = None
if FACT_AFF_CSV.exists():
    df_aff = pd.read_csv(FACT_AFF_CSV, low_memory=False,
                          usecols=lambda c: c in JOIN_KEYS or c in ('affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all'))
    aff_lookup = df_aff
    print(f'factuality_affiliation rows: {len(df_aff):,}')

def aff_for(row, name, lastname):
    if aff_lookup is None:
        return {}
    sub = aff_lookup[(aff_lookup['model']==row['model']) & (aff_lookup['language']==row['language']) &
                     (aff_lookup['run_id']==row['run_id']) & (aff_lookup['name']==name) & (aff_lookup['lastname']==lastname) &
                     (aff_lookup['role']==row['role']) & (aff_lookup['task']==row['task']) & (aff_lookup['location']==row['location']) &
                     (aff_lookup['field']==row['field']) & (aff_lookup['subfield']==row['subfield'])]
    if len(sub) == 0:
        return {}
    r = sub.iloc[0]
    return {
        'affiliation_status_auto':         r.get('affiliation_status'),
        'affiliation_best_match_score':    r.get('affiliation_best_match_score'),
        'affiliation_best_match_oa':       r.get('affiliation_best_match_oa'),
        'affiliation_oa_all':              r.get('affiliation_oa_all'),
    }

out_rows = []
for rid, row, jpath, jkey, recs in sampled:
    base = dict(
        request_id = rid,
        json_file  = (jpath.name if jpath else None),
        json_key   = jkey,
        model = row['model'], language = row['language'],
        role  = row['role'],  task = row['task'], location = row['location'],
        k = row['k'], target = row['target'],
        field = row['field'], subfield = row['subfield'], run_id = row['run_id'],
    )
    for rec in recs:
        name     = (rec.get('name') or '').strip()
        lastname = (rec.get('lastname') or '').strip()
        cur_aff  = rec.get('current_affiliations')
        areas    = rec.get('areas_of_research_or_work')
        reason   = rec.get('reason')
        source   = rec.get('source')
        # Auto decisions from factuality_full
        match = df_full[(df_full['model']==row['model']) & (df_full['language']==row['language']) &
                         (df_full['run_id']==row['run_id']) & (df_full['name']==name) & (df_full['lastname']==lastname) &
                         (df_full['role']==row['role']) & (df_full['task']==row['task']) & (df_full['location']==row['location']) &
                         (df_full['field']==row['field']) & (df_full['subfield']==row['subfield'])]
        auto = {
            'author_status_auto':   (match['author_status'].iloc[0]   if len(match) else None),
            'oa_status_auto':       (match['oa_status'].iloc[0]       if len(match) else None),
            'field_status_auto':    (match['field_status'].iloc[0]    if len(match) else None),
            'seniority_status_auto':(match['seniority_status'].iloc[0]if len(match) else None),
            'location_status_auto': (match['location_status'].iloc[0] if len(match) else None),
            'matched_name':         (match['matched_name'].iloc[0]    if len(match) else None),
            'researcher_id':        (match['researcher_id'].iloc[0]   if len(match) else None),
            'match_score':          (match['match_score'].iloc[0]     if len(match) else None),
            'gt_field':             (match['gt_field'].iloc[0]        if len(match) else None),
            'oa_id':                (match['oa_id'].iloc[0]           if len(match) else None),
            'oa_display_name':      (match['oa_display_name'].iloc[0]  if len(match) else None),
            'oa_match_score':       (match['oa_match_score'].iloc[0]   if len(match) else None),
        }
        auto.update(aff_for(row, name, lastname))
        # Search URL hints for the human reviewer
        q = quote(f'{name} {lastname}'.strip())
        out_rows.append({
            **base,
            'name': name, 'lastname': lastname,
            'current_affiliations': cur_aff, 'areas_of_research_or_work': areas,
            'reason': reason, 'source': source,
            **auto,
            'ss_search_url': f'https://www.semanticscholar.org/search?q={q}',
            'oa_search_url': f'https://api.openalex.org/authors?search={q}',
            # EMPTY columns to fill manually:
            'found_in_ss_manual':         '',
            'found_in_oa_manual':         '',
            'affiliation_correct_manual': '',
            'field_correct_manual':       '',
            'notes_manual':               '',
        })

df_out = pd.DataFrame(out_rows)
df_out.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df_out)} persona rows for {N_REQUESTS} requests → {OUT_CSV}')
df_out[['request_id','name','lastname','author_status_auto'] + (['affiliation_status_auto'] if 'affiliation_status_auto' in df_out.columns else [])]

In [ ]:
# 4. Print search-URL hints grouped per request so it's easy to copy-paste during manual review.
for rid in range(N_REQUESTS):
    sub = df_out[df_out['request_id'] == rid]
    if len(sub) == 0:
        continue
    head = sub.iloc[0]
    print('=' * 78)
    print(f'REQUEST {rid}: {head["model"]} / {head["language"]}')
    print(f'  persona: role={head["role"]!r} task={head["task"]!r} location={head["location"]!r}')
    print(f'  request: k={head["k"]} target={head["target"]!r} field={head["field"]!r} subfield={head["subfield"]!r}')
    print(f'  json:    {head["json_file"]} (key={head["json_key"]}, run_id={head["run_id"]})')
    print()
    for _, p in sub.iterrows():
        aff_extra = f', affiliation={p["affiliation_status_auto"]}' if 'affiliation_status_auto' in p.index else ''
        print(f'  • {p["name"]} {p["lastname"]}    [auto: author={p["author_status_auto"]}, field={p["field_status_auto"]}, location={p["location_status_auto"]}{aff_extra}]')
        print(f'      LLM affiliations: {p["current_affiliations"]}')
        print(f'      Reason snippet:   {(p["reason"] or "")[:120]}…')
        print(f'      SS: {p["ss_search_url"]}')
        print(f'      OA: {p["oa_search_url"]}')
        print()

## Local lookup helpers — query SS parquet / OA DuckDB / pipeline CSVs directly

Faster than opening browser tabs: copy a `(name, lastname)` from `manual_validation_20.csv` and run `validate(...)` to see all three sources side-by-side.

- **SS** (`Researchers_Deduplicated_Genderize_Namsor.parquet`): the same GT used by `factuality_author_jw.py`.
- **OA** (`openalex_latest.duckdb`): same DB used by `factuality_openalex.py` / `factuality_affiliation.py`.
- **Pipeline** (`factuality_full.csv`): the auto decisions, joined per persona.


In [ ]:
# Local lookup helpers — SS parquet, OA DuckDB, pipeline CSV
# ───────────────────────────────────────────────────────────────────────
import re
import duckdb
import pandas as pd
from IPython.display import display

SS_PARQUET = '/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet'
OA_DB      = '/data/datasets/LLMScholar-Personas/data/openalex_latest.duckdb'

# Load SS once (kept in memory, ~216 MB).
_df_ss = pd.read_parquet(SS_PARQUET)
print(f'SS loaded: {len(_df_ss):,} rows')

# This parquet has `Name` as the full name (no separate LastName).
# `Clean_standarized_name` is normalized (lowercase, no accents).
_SS_FULL_COL  = 'Name' if 'Name' in _df_ss.columns else None
_SS_CLEAN_COL = 'Clean_standarized_name' if 'Clean_standarized_name' in _df_ss.columns else None
_SS_FIELD_COL = 'Field' if 'Field' in _df_ss.columns else None
print(f'  full-name col: {_SS_FULL_COL!r}   clean col: {_SS_CLEAN_COL!r}')

# Pre-normalize for case/accent-insensitive search
import unicodedata
def _norm(s: str) -> str:
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return s.lower()

# Normalized cache of the names (single pass).
if _SS_FULL_COL:
    _ss_name_norm = _df_ss[_SS_FULL_COL].astype(str).map(_norm)
    print(f'  cached _ss_name_norm: {len(_ss_name_norm):,} values')

# OA connection in read-only mode.
_oa_con = duckdb.connect(OA_DB, read_only=True)

# Load factuality_full once for lookup of automatic decisions.
_df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False)
print(f'factuality_full loaded: {len(_df_full):,} rows')


def find_ss(name: str, lastname: str, limit: int = 20) -> pd.DataFrame:
    """Search the SS parquet by substring of '<name> <lastname>'.
    Case- and accent-insensitive."""
    if _SS_FULL_COL is None:
        return pd.DataFrame()
    n  = _norm(name)
    ln = _norm(lastname)
    # Both first name and last name must appear (in any order)
    mask = _ss_name_norm.str.contains(re.escape(n),  na=False) & \
           _ss_name_norm.str.contains(re.escape(ln), na=False)
    cols = [c for c in (_SS_FULL_COL, _SS_FIELD_COL, 'Researcher_id',
                        'Combined_gender', 'Citations', 'Productivity',
                        'First_year', 'Last_year')
            if c and c in _df_ss.columns]
    return _df_ss.loc[mask, cols].head(limit)


def find_oa(name: str, lastname: str, limit: int = 10) -> pd.DataFrame:
    """Search for an author in the OpenAlex DuckDB by name + lastname.

    Searches against display_name AND all display_name_alternatives (same as
    the pipeline). Optimized with list_filter to avoid the UNNEST blow-up
    over 113M authors — takes ~4s per query vs ~100s with UNNEST."""
    n  = name.replace("'", "''").lower()
    ln = lastname.replace("'", "''").lower()
    q = f"""
        WITH filtered AS (
          SELECT id, display_name, works_count, cited_by_count, last_known_institution,
                 list_filter(list_concat([display_name], COALESCE(display_name_alternatives, [])),
                             x -> LOWER(x) LIKE '%{n}%' AND LOWER(x) LIKE '%{ln}%') AS matches
          FROM authors
        )
        SELECT id, display_name, works_count, cited_by_count, last_known_institution,
               matches[1] AS matched_via
        FROM filtered WHERE len(matches) > 0
        ORDER BY cited_by_count DESC NULLS LAST
        LIMIT {limit}
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA query failed: {exc}')
        return pd.DataFrame()


_WORKS_AGG_GLOB = '/data/asanchez/duckdb_enrich/oa_works_agg_chunks/chunk_*.parquet'

def oa_institutions(oa_id: str) -> pd.DataFrame:
    """Institution history for an author.
    Uses the pre-aggregated chunks (same data source as factuality_affiliation.py)
    instead of doing UNNEST over the full `works` table — the latter kills the kernel."""
    q = f"""
        SELECT inst_name AS institution, country, MIN(last_year) AS first_year, MAX(last_year) AS last_year
          FROM read_parquet('{_WORKS_AGG_GLOB}')
         WHERE oa_id = '{oa_id}'
           AND inst_name IS NOT NULL
         GROUP BY inst_name, country
         ORDER BY last_year DESC
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA institutions query failed: {exc}')
        return pd.DataFrame()


def lookup_pipeline(name: str, lastname: str) -> pd.DataFrame:
    """What each pipeline step said about this persona."""
    q = _df_full[
        (_df_full['name'].astype(str).str.lower()     == name.lower()) &
        (_df_full['lastname'].astype(str).str.lower() == lastname.lower())
    ]
    cols = [c for c in ('model','language','role','task','location','field','subfield',
                        'author_status','field_status','seniority_status','location_status',
                        'affiliation_status','oa_id','oa_country_code','oa_last_institution')
            if c in q.columns]
    return q[cols].drop_duplicates().head(20)


def validate(name: str, lastname: str) -> None:
    """Print SS + OA + pipeline for a persona."""
    print(f'═══ {name} {lastname} ═══')
    print('\n— Semantic Scholar (parquet):')
    ss = find_ss(name, lastname)
    if len(ss):
        display(ss)
    else:
        print('  (no matches)')
    print('\n— OpenAlex (DuckDB, ordered by citations):')
    oa = find_oa(name, lastname)
    if len(oa):
        display(oa)
        top_id = oa.iloc[0]['id']
        print(f'\n— OA institution history for top hit ({top_id}):')
        display(oa_institutions(top_id))
    else:
        print('  (no matches)')
    print('\n— Pipeline (factuality_full.csv):')
    pp = lookup_pipeline(name, lastname)
    if len(pp):
        display(pp)
    else:
        print('  (not in factuality_full)')


# Example:
# validate('Maria', 'Gonzalez')


### Helper end-to-end — `validate_row(row)`

Takes a row from the CSV and shows you **everything in a single output**: the persona prompt, what the LLM said, what SS says, what OA says (institutions + topics), and the pipeline's verdict. At the end it prints a **suggestion** for each manual column.

Usage:
```python
for i, row in df_out.iterrows():
    validate_row(row)
    input('Press Enter to continue…')   # optional: pause between rows
```


In [ ]:
# End-to-end row validator
# ─────────────────────────────────────────────────────────────────────────
def oa_author_topics(oa_id: str, limit: int = 5) -> pd.DataFrame:
    """Top topics for the author in OA. If the column doesn't exist in this dump,
    returns an empty DataFrame without raising."""
    for col_path in ('UNNEST(a.topics) AS u(t)', 'UNNEST(a.x_concepts) AS u(t)'):
        q = f"""
            SELECT t.display_name AS topic,
                   t.field.display_name AS field,
                   t.subfield.display_name AS subfield,
                   t.count
              FROM authors AS a, {col_path}
             WHERE a.id = '{oa_id}'
             ORDER BY t.count DESC
             LIMIT {limit}
        """
        try:
            df = _oa_con.execute(q).fetchdf()
            if len(df):
                return df
        except Exception:
            continue
    return pd.DataFrame()


def _norm_eq(a, b) -> bool:
    return _norm(str(a)) == _norm(str(b)) if a and b else False


def validate_row(row) -> None:
    """Print full context + suggestions for each manual column."""
    print('═' * 70)
    print(f"REQUEST {row['request_id']} — {row['name']} {row['lastname']}")
    print('═' * 70)

    # 1. What the persona prompt asked for
    print('\n┌─ PROMPT sent to LLM ─')
    print(f"│  model       = {row['model']}  ({row['language']})")
    print(f"│  role        = {row['role']}")
    print(f"│  task        = {row['task']}     location = {row['location']}")
    print(f"│  field       = {row['field']}")
    print(f"│  subfield    = {row['subfield']}")

    # 2. What the LLM said about this persona
    print('\n┌─ LLM said ─')
    print(f"│  name             = {row['name']} {row['lastname']}")
    print(f"│  current_affil.   = {row['current_affiliations']}")
    print(f"│  areas_of_work    = {row['areas_of_research_or_work']}")
    print(f"│  reason snippet   = {str(row.get('reason',''))[:120]}…")

    # 3. Pipeline verdict
    print('\n┌─ PIPELINE automatic verdict ─')
    for c in ('author_status_auto','oa_status_auto','field_status_auto','seniority_status_auto',
              'location_status_auto','affiliation_status_auto',
              'affiliation_best_match_score','affiliation_best_match_oa'):
        if c in row.index and pd.notna(row[c]) and row[c] != '':
            print(f"│  {c:32s} = {row[c]}")

    # 3.5 Who the pipeline matched in SS and OA (what was already saved)
    print('\n┌─ PIPELINE matches (SS + OA, what the pipeline found) ─')
    if 'matched_name' in row.index and pd.notna(row.get('matched_name')) and row['matched_name'] != '':
        print(f"│  SS matched_name    = {row['matched_name']}  (researcher_id={row.get('researcher_id')}, score={row.get('match_score')}, gt_field={row.get('gt_field')})")
    else:
        print(f"│  SS                  → no match in SS parquet")
    if 'oa_id' in row.index and pd.notna(row.get('oa_id')) and row['oa_id'] != '':
        score = row.get('oa_match_score')
        kind = 'exact' if pd.notna(score) and float(score) == 1.0 else f'fuzzy ({score})'
        print(f"│  OA display_name    = {row['oa_display_name']}  (oa_id={row['oa_id']}, {kind})")
    else:
        print(f"│  OA                  → no match in OpenAlex")

    # 4. SS — search and show ALL matches (you pick which one)
    print('\n┌─ Semantic Scholar (parquet) ─')
    ss = find_ss(row['name'], row['lastname'])
    if len(ss):
        print(ss.to_string(index=False))
    else:
        print('  (no matches — try removing initials or particles)')

    # 5. OA — other authors with similar name (for the reviewer to browse, not the pipeline's match)
    print('\n┌─ OpenAlex (DuckDB, other similar ones, top 5 by citations) ─')
    oa = find_oa(row['name'], row['lastname'], limit=5)
    if len(oa):
        print(oa.to_string(index=False))
        top_id = oa.iloc[0]['id']
        print(f'\n  Institutions for top hit ({top_id}):')
        inst = oa_institutions(top_id)
        if len(inst):
            print('  ' + inst.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (no institutions)')
        print(f'\n  Topics/Fields for top hit:')
        topics = oa_author_topics(top_id)
        if len(topics):
            print('  ' + topics.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (no topics — or the column does not exist in this dump)')
    else:
        print('  (no matches)')

    # 6. Suggestions for filling the manual columns
    print('\n┌─ SUGGESTIONS for filling the CSV ─')
    print(f"│  found_in_ss_manual         → 'yes' if you see a clear match above, 'no' if the list is empty or all are a different person")
    print(f"│  found_in_oa_manual         → same for OA")
    print(f"│  field_correct_manual       → compare requested field ({row['field']!r}) with SS's Field or OA's Topics")
    print(f"│  affiliation_correct_manual → does the LLM's affiliation ({row['current_affiliations']}) appear in the OA institution list above?")
    print(f"│  notes_manual               → free text (e.g. 'same name but different field')")
    print()


## Manual review step

Open `manual_validation_20.csv` in Excel/LibreOffice and fill in the empty columns for each persona row:

- `found_in_ss_manual`: `yes` / `no` / empty (unknown). Did you find this exact researcher on Semantic Scholar?
- `found_in_oa_manual`: same, for OpenAlex.
- `affiliation_correct_manual`: `yes` / `no` / empty. Does at least one institution in `current_affiliations` match a real affiliation of this researcher?
- `field_correct_manual`: `yes` / `no` / empty. Does the researcher actually work in the requested `field`?
- `notes_manual`: free text — record anything notable (e.g. "same name but different person", "affiliation outdated").

When done, re-run the cell below to compute concordance.

In [ ]:
# 5. Concordance after manual filling. Re-run after editing the CSV.
df_filled = pd.read_csv('/home/asanchez/code/asanchez/LLMScholar-Personas/results/results/factualities_v2/manual/manual_validation_20_v2_filled.csv', sep=',', dtype=str).fillna('')

# ── Enrich with oa_status from factuality_full.csv (no need to regenerate CSV) ──
# If the manual CSV doesn't have `oa_status_auto`, we pull it from the pipeline join-on
# (model, language, role, task, location, k, target, field, subfield, run_id, name, lastname).
if 'oa_status_auto' not in df_filled.columns:
    join_keys = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
    print(f'Pulling oa_status from factuality_full …')
    df_pipe = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                          usecols=lambda c: c in join_keys + ['oa_status'])
    df_pipe = df_pipe.dropna(subset=join_keys).drop_duplicates(subset=join_keys)
    # Cast keys to str to join with df_filled (which is dtype=str)
    for k in join_keys:
        df_pipe[k] = df_pipe[k].astype(str)
    df_filled = df_filled.merge(df_pipe.rename(columns={'oa_status': 'oa_status_auto'}),
                                on=join_keys, how='left')
    df_filled['oa_status_auto'] = df_filled['oa_status_auto'].fillna('')
    n_with_oa = (df_filled['oa_status_auto'] != '').sum()
    print(f'  oa_status_auto filled in {n_with_oa}/{len(df_filled)} rows')


def truthy(v):
    return str(v).strip().lower() in ('yes','y','true','1','t')

def manual_found_any_row(r):
    """True/False/None: True if found in SS or OA; False if 'no' in both; None if empty."""
    ss = r['found_in_ss_manual']
    oa = r['found_in_oa_manual']
    if not ss and not oa:
        return None
    return truthy(ss) or truthy(oa)

def auto_found_row(r):
    """auto_found = found in SS OR found in OA (symmetric with the manual criterion)."""
    ss = r.get('author_status_auto', '')
    oa = r.get('oa_status_auto', '')
    if not ss and not oa:
        return None
    return (ss == 'found') or (oa == 'found')

df_filled['manual_found_any'] = df_filled.apply(manual_found_any_row, axis=1)
df_filled['auto_found']       = df_filled.apply(auto_found_row,       axis=1)

# ── Author concordance (auto = SS OR OA) ──────────────────────────────────────
both_labeled = df_filled[df_filled['manual_found_any'].notna() & df_filled['auto_found'].notna()]
n_compared = len(both_labeled)
n_agree    = (both_labeled['manual_found_any'] == both_labeled['auto_found']).sum()
if n_compared > 0:
    print(f'\nAUTHOR FOUND concordance (auto = SS OR OA): {n_agree}/{n_compared} = {100*n_agree/n_compared:.1f}%')
    print()
    print('Confusion (auto_found × manual_found_any):')
    print(both_labeled.groupby(['auto_found','manual_found_any']).size().unstack(fill_value=0))
else:
    print('No manual labels filled yet — fill the CSV then re-run.')

# Per-request breakdown
if n_compared > 0:
    by_req = both_labeled.groupby('request_id').apply(
        lambda g: pd.Series({
            'n_personas': len(g),
            'agree': (g['manual_found_any'] == g['auto_found']).sum(),
        }),
        include_groups=False,
    )
    by_req['pct'] = (100 * by_req['agree'] / by_req['n_personas']).round(1)
    print()
    print('Per request:')
    print(by_req)

# ── Affiliation concordance — honest version ─────────────────────────────────
if 'affiliation_status_auto' in df_filled.columns:
    real_found = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['affiliation_correct_manual'].str.strip() != '') &
        (df_filled['affiliation_status_auto'].str.strip()     != '')
    ].copy()

    if len(real_found) > 0:
        real_found['auto_aff_match']   = real_found['affiliation_status_auto'] == 'affiliation_match'
        real_found['manual_aff_match'] = real_found['affiliation_correct_manual'].apply(truthy)
        n_aff   = len(real_found)
        ag_aff  = (real_found['auto_aff_match'] == real_found['manual_aff_match']).sum()
        print()
        print(f'AFFILIATION concordance (only authors actually found): {ag_aff}/{n_aff} = {100*ag_aff/n_aff:.1f}%')
        print()
        print('Confusion (auto_aff_match × manual_aff_match):')
        print(real_found.groupby(['auto_aff_match','manual_aff_match']).size().unstack(fill_value=0))
    else:
        print()
        print('AFFILIATION concordance: no authors actually found with affiliation labeled.')

# ── Field concordance — same honest logic ────────────────────────────────────
if 'field_status_auto' in df_filled.columns and (df_filled['field_correct_manual'].str.strip() != '').any():
    real_found_field = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['field_correct_manual'].str.strip()  != '') &
        (df_filled['field_status_auto'].str.strip()      != '')
    ].copy()
    if len(real_found_field) > 0:
        real_found_field['auto_field_match']   = real_found_field['field_status_auto'] == 'field_match'
        real_found_field['manual_field_match'] = real_found_field['field_correct_manual'].apply(truthy)
        n_f  = len(real_found_field)
        ag_f = (real_found_field['auto_field_match'] == real_found_field['manual_field_match']).sum()
        print()
        print(f'FIELD concordance (only authors actually found): {ag_f}/{n_f} = {100*ag_f/n_f:.1f}%')


## Pipeline metrics against the manual review

We take the manual annotations as **ground truth** and measure how much the automatic pipeline agrees with them. For each dimension we define what is "positive":

| Dimension | Auto-positive means | Manual-positive means |
|---|---|---|
| **author_found** | `author_status='found'` OR `oa_status='found'` | `found_in_ss_manual='yes'` OR `found_in_oa_manual='yes'` |
| **affiliation_match** | `affiliation_status='affiliation_match'` | `affiliation_correct_manual='yes'` |
| **field_match** | `field_status='field_match'` | `field_correct_manual='yes'` |

Metrics reported per dimension:

- **Accuracy** = (TP + TN) / N — total percentage correct (positives and negatives).
- **Precision** = TP / (TP + FP) — of the rows where auto said "positive", how many were really positive.
- **Recall** = TP / (TP + FN) — of the actually positive rows, how many auto detected.
- **F1** = harmonic mean of precision and recall (balance between the two).

For each dimension we print:
1. The **confusion matrix** 2×2 (auto × manual).
2. The 4 metrics in absolute terms (numerator/denominator) and percentage.

Quick interpretation:
- *High precision, low recall* → auto is **cautious**: when it says "positive" it usually gets it right, but it misses cases.
- *Low precision, high recall* → auto is **liberal**: it marks many positives but includes false positives.
- *Both high* → the classifier works well.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Pipeline auto metrics vs. manual annotation (ground truth)
# See explanation of each dimension in the previous cell.
# ────────────────────────────────────────────────────────────────────────────
def truthy(v):
    return str(v).strip().lower() in ('yes', 'y', 'true', '1', 't')

def has_text(s):
    return s.fillna('').astype(str).str.strip() != ''


def report(dim_name, auto_pos, manual_pos, mask):
    """Print confusion matrix + metrics for a dimension."""
    auto_pos   = auto_pos[mask].astype(bool).reset_index(drop=True)
    manual_pos = manual_pos[mask].astype(bool).reset_index(drop=True)
    n = len(auto_pos)
    if n == 0:
        print(f"── {dim_name} ──")
        print(f"  (no rows with manual annotation; nothing to compute)\n")
        return None

    TP = int(( auto_pos &  manual_pos).sum())
    FP = int(( auto_pos & ~manual_pos).sum())
    FN = int((~auto_pos &  manual_pos).sum())
    TN = int((~auto_pos & ~manual_pos).sum())

    prec = TP / (TP + FP) if (TP + FP) else float("nan")
    rec  = TP / (TP + FN) if (TP + FN) else float("nan")
    acc  = (TP + TN) / n
    f1   = (2 * prec * rec / (prec + rec)) if (prec == prec and rec == rec and (prec + rec)) else float("nan")

    def pct(x):
        return f"{100*x:.1f}%" if x == x else "N/A"

    print(f"── {dim_name} ──")
    print(f"  Comparable rows: {n}")
    print()
    print(f"  Confusion matrix:")
    print(f"                           manual positive   manual negative")
    print(f"     auto positive  →  TP={TP:<3}             FP={FP}")
    print(f"     auto negative  →  FN={FN:<3}             TN={TN}")
    print()
    print(f"  Accuracy  = (TP+TN)/N  = ({TP}+{TN})/{n} = {pct(acc)}")
    print(f"  Precision = TP/(TP+FP) = {TP}/{TP+FP}{'  = ' + pct(prec) if (TP+FP) else ' = N/A (auto marked no positives)'}")
    print(f"  Recall    = TP/(TP+FN) = {TP}/{TP+FN}{'  = ' + pct(rec) if (TP+FN) else ' = N/A (no real positives)'}")
    print(f"  F1        = {pct(f1)}")
    print()
    return {"dim": dim_name, "n": n, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "accuracy_%": round(100*acc,1), "precision_%": round(100*prec,1) if prec==prec else None,
            "recall_%": round(100*rec,1) if rec==rec else None,
            "f1_%": round(100*f1,1) if f1==f1 else None}


# ── 1) author_found ──
auto_author   = (df_filled["author_status_auto"] == "found") | (df_filled["oa_status_auto"] == "found")
manual_author = df_filled["found_in_ss_manual"].apply(truthy) | df_filled["found_in_oa_manual"].apply(truthy)
mask_author   = has_text(df_filled["found_in_ss_manual"]) | has_text(df_filled["found_in_oa_manual"])
r1 = report("author_found  (does the person exist?)", auto_author, manual_author, mask_author)

# ── 2) affiliation_match ──
auto_aff   = df_filled.get("affiliation_status_auto", pd.Series("", index=df_filled.index)) == "affiliation_match"
manual_aff = df_filled["affiliation_correct_manual"].apply(truthy)
mask_aff   = has_text(df_filled["affiliation_correct_manual"])
r2 = report("affiliation_match  (does the affiliation match?)", auto_aff, manual_aff, mask_aff)

# ── 3) field_match ──
auto_field   = df_filled.get("field_status_auto", pd.Series("", index=df_filled.index)) == "field_match"
manual_field = df_filled["field_correct_manual"].apply(truthy)
mask_field   = has_text(df_filled["field_correct_manual"])
r3 = report("field_match  (do they work in that field?)", auto_field, manual_field, mask_field)

# ── Compact final summary ──
print("═" * 70)
print("SUMMARY")
print("═" * 70)
summary = pd.DataFrame([r for r in (r1, r2, r3) if r is not None])
if len(summary):
    display(summary[["dim", "n", "TP", "FP", "FN", "TN", "accuracy_%", "precision_%", "recall_%", "f1_%"]])


---

## Alternative sample: 10 random recommendations

Direct sampling on `factuality_full.csv` (1 row = 1 recommendation, not grouped by request). Simpler than the 20-requests flow: it doesn't load JSONs or reconstruct personas — it uses what the pipeline already saved. Uses the `summary_v2/factuality_full.csv` loaded above.

Set `RANDOM_STATE_RECS = None` for a different sample each run; with an int it stays reproducible.

In [ ]:
# ── Generate manual_validation_10_random.csv ─────────────────────────────────
from urllib.parse import quote

OUT_RANDOM = OUT_DIR / 'manual_validation_10_random_v2.csv'

N_RECS             = 10
RANDOM_STATE_RECS  = 42        # None for a different sample each run

KEEP_COLS = [
    'model','language','role','task','location','k','target','field','subfield','run_id',
    'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
    'author_status','oa_status','field_status','seniority_status','location_status',
    'affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
    'matched_name','researcher_id','match_score',
    'oa_id','oa_display_name','oa_match_score',
    'gt_field','gt_citations',
]
df_v2 = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                    usecols=lambda c: c in KEEP_COLS)
print(f'factuality_full v2 rows: {len(df_v2):,}')

sample = df_v2.sample(n=N_RECS, random_state=RANDOM_STATE_RECS).reset_index(drop=True)

# Rename to *_auto to align with the 20-requests CSV
rename_auto = {
    'author_status':      'author_status_auto',
    'oa_status':          'oa_status_auto',
    'field_status':       'field_status_auto',
    'seniority_status':   'seniority_status_auto',
    'location_status':    'location_status_auto',
    'affiliation_status': 'affiliation_status_auto',
}
sample = sample.rename(columns=rename_auto)

# Search URLs for the human reviewer
def _q(n, l): return quote(f'{n} {l}'.strip())
sample['ss_search_url'] = sample.apply(lambda r: f'https://www.semanticscholar.org/search?q={_q(r["name"], r["lastname"])}', axis=1)
sample['oa_search_url'] = sample.apply(lambda r: f'https://api.openalex.org/authors?search={_q(r["name"], r["lastname"])}', axis=1)

# Empty manual columns
for c in ('found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual'):
    sample[c] = ''

# Final order
ordered = (
    ['model','language','role','task','location','k','target','field','subfield','run_id',
     'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
     'author_status_auto','oa_status_auto','field_status_auto','seniority_status_auto','location_status_auto',
     'affiliation_status_auto','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
     'matched_name','researcher_id','match_score','gt_field','gt_citations',
     'oa_id','oa_display_name','oa_match_score',
     'ss_search_url','oa_search_url',
     'found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual']
)
sample = sample[[c for c in ordered if c in sample.columns]]

sample.to_csv(OUT_RANDOM, index=False)
print(f'Wrote {len(sample)} random recommendations → {OUT_RANDOM}')
sample[['name','lastname','model','language','field','author_status_auto','oa_status_auto','field_status_auto','affiliation_status_auto']]
